In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!ls "/content/drive/MyDrive/Colab Notebooks/ProjectImagePro/Classify_Lung/runs/classify/train/weights/"


ls: cannot access '/content/drive/MyDrive/Colab Notebooks/ProjectImagePro/Classify_Lung/runs/classify/train/weights/': No such file or directory


In [ ]:
#PREPROCESSING
import os
import cv2
import numpy as np
from tqdm import tqdm

base_path = "/content/drive/MyDrive/Colab Notebooks/ProjectImagePro/Classify_Lung"
output_path = os.path.join(base_path, "processed_simple")
os.makedirs(output_path, exist_ok=True)


#Preprocess

def simple_preprocess(image_path, size=(640, 640)):

    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None

    # ปรับ contrast
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    img = clahe.apply(img)

    # ลด noise
    img = cv2.GaussianBlur(img, (3, 3), 0)

    # Resize
    img = cv2.resize(img, size)

    # Normalize INTENSITY 0-255
    img = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX)

    return img

#train/val/test

folders = ["train", "val", "test"]
classes = ["normal", "pneumonia"]

for folder in folders:
    for cls in classes:
        input_dir = os.path.join(base_path, folder, cls)
        output_dir = os.path.join(output_path, folder, cls)
        os.makedirs(output_dir, exist_ok=True)

        print(f"🔹 Processing {folder}/{cls} ...")
        for file in tqdm(os.listdir(input_dir)):
            if not file.lower().endswith(('.jpg', '.jpeg', '.png')):
                continue
            in_path = os.path.join(input_dir, file)
            out_path = os.path.join(output_dir, file)

            processed = simple_preprocess(in_path)
            if processed is not None:
                cv2.imwrite(out_path, processed)

print("Preprocessing complete! Results saved to:")
print(output_path)


🔹 Processing train/normal ...


100%|██████████| 1341/1341 [03:58<00:00,  5.62it/s]


🔹 Processing train/pneumonia ...


100%|██████████| 3875/3875 [05:05<00:00, 12.69it/s]


🔹 Processing val/normal ...


...

🔹 Processing test/normal ...


100%|██████████| 234/234 [00:21<00:00, 10.70it/s]


🔹 Processing test/pneumonia ...


100%|██████████| 390/390 [00:22<00:00, 17.56it/s]


✅ Preprocessing complete! Results saved to:
/content/drive/MyDrive/Colab Notebooks/ProjectImagePro/Classify_Lung/processed_simple


In [ ]:
folders = ["train", "val", "test"]
classes = ["normal", "pneumonia"]

base_path = "/content/drive/MyDrive/Colab Notebooks/ProjectImagePro/Classify_Lung"
output_path = os.path.join(base_path, "balanced_dataset_v2")
for folder in folders:
    for cls in classes:
        output_dir = os.path.join(output_path, folder, cls)
        open(os.path.join(output_dir, ".keep"), "w").close()


In [ ]:

#AUGMENTATION
import os
import cv2
import numpy as np
from tqdm import tqdm
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import shutil


input_base = "/content/drive/MyDrive/Colab Notebooks/ProjectImagePro/Classify_Lung/balanced_dataset_v2"
output_base = "/content/drive/MyDrive/Colab Notebooks/ProjectImagePro/Classify_Lung/augmented_simple"
os.makedirs(output_base, exist_ok=True)


#Data Generator

datagen = ImageDataGenerator(
    rotation_range=10,       # หมุน ±10 องศา
    width_shift_range=0.1,   # ขยับแนวนอน
    height_shift_range=0.1,  # ขยับแนวตั้ง
    zoom_range=0.1,          # ซูม ±10%
    brightness_range=[0.9, 1.1],  # ปรับแสง ±10%
    horizontal_flip=True,    # พลิกซ้ายขวา
    fill_mode='nearest'
)

#train

from tensorflow.keras.preprocessing import image
import numpy as np

train_dir = os.path.join(input_base, "train")
output_train = os.path.join(output_base, "train")
os.makedirs(output_train, exist_ok=True)

for cls in ["normal", "pneumonia"]:
    input_folder = os.path.join(train_dir, cls)
    output_folder = os.path.join(output_train, cls)
    os.makedirs(output_folder, exist_ok=True)

    print(f" Augmenting {cls} ...")
    for file in tqdm(os.listdir(input_folder)):
        if not file.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue

        img_path = os.path.join(input_folder, file)
        img = image.load_img(img_path, target_size=(224,224))
        x = image.img_to_array(img)
        x = np.expand_dims(x, axis=0)

        # สร้างภาพใหม่ 3 รูปต่อ 1 รูปต้นฉบับ
        i = 0
        for batch in datagen.flow(x, batch_size=1,
                                  save_to_dir=output_folder,
                                  save_prefix="aug",
                                  save_format="jpg"):
            i += 1
            if i >= 3:
                break  # สร้างแค่ 3 รูปต่อภาพ

print("\nAugmentation complete!")
print(f"Results saved to: {output_base}")


 Augmenting normal ...


100%|██████████| 1342/1342 [03:32<00:00,  6.33it/s]


 Augmenting pneumonia ...


100%|██████████| 1342/1342 [03:15<00:00,  6.87it/s]


Augmentation complete!
Results saved to: /content/drive/MyDrive/Colab Notebooks/ProjectImagePro/Classify_Lung/augmented_simple


In [ ]:
import shutil

# path ของโฟลเดอร์ต้นฉบับ
folder_path = "/content/drive/MyDrive/Colab Notebooks/ProjectImagePro/Classify_Lung/augmented_simple"

# path สำหรับบันทึกไฟล์ .zip (ไม่ต้องใส่ .zip ตอนท้าย)
zip_path = "/content/drive/MyDrive/Colab Notebooks/ProjectImagePro/Classify_Lung/augmented_simple"

# สร้าง zip file
shutil.make_archive(zip_path, 'zip', folder_path)

print("สำเร็จ! สร้างไฟล์ zip แล้วที่:")
print(zip_path + ".zip")


✅ สำเร็จ! สร้างไฟล์ zip แล้วที่:
/content/drive/MyDrive/Colab Notebooks/ProjectImagePro/Classify_Lung/augmented_simple.zip
